In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Cell 1

In [2]:
!pip install -q "numpy==1.26.4"
!pip install -q "opencv-python-headless==4.10.0.84"
!pip install -q "albumentations==1.4.24"
!pip install -q "segmentation-models-pytorch==0.3.4"
!pip install -q tqdm matplotlib pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 67.0 MB/s eta 0:00:00
ERROR: Operation cancelled by user
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 20.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.6 MB/s eta 0:00:00


In [3]:
!pip install -q --no-deps segmentation-models-pytorch==0.3.4
!pip install -q --no-deps timm==0.9.7
!pip install -q --no-deps efficientnet-pytorch==0.7.1
!pip install -q --no-deps pretrainedmodels==0.7.4

In [4]:
import sys, site, shutil, glob, os

# Xoá sạch numpy cũ/bị lỗi trong site-packages
for sp in site.getsitepackages():
    for pattern in ["numpy", "numpy-*", "~umpy*"]:
        for path in glob.glob(os.path.join(sp, pattern)):
            print("Removing:", path)
            shutil.rmtree(path, ignore_errors=True)

print("Cleaned old numpy folders")

Removing: /usr/local/lib/python3.12/dist-packages/numpy
Removing: /usr/local/lib/python3.12/dist-packages/numpy-1.26.4.dist-info
Removing: /usr/local/lib/python3.12/dist-packages/~umpy.libs
Removing: /usr/local/lib/python3.12/dist-packages/~umpy
Removing: /usr/local/lib/python3.12/dist-packages/~umpy-2.0.2.dist-info
Cleaned old numpy folders


In [5]:
!pip install -q --no-cache-dir --force-reinstall "numpy==2.0.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 178.5 MB/s eta 0:00:00


In [6]:
import numpy as np
import numpy.random
import torch

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

# Test optimizer
test_model = torch.nn.Linear(10, 1)
test_optimizer = torch.optim.AdamW(test_model.parameters(), lr=1e-4)

x = torch.randn(2, 10)
y = test_model(x).sum()
y.backward()
test_optimizer.step()

print("Optimizer test: OK")

NumPy: 2.0.2
Torch: 2.10.0+cu128
CUDA: True
Optimizer test: OK


Cell 3

In [7]:
import os
import json
import time
import random
from pathlib import Path

from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


Cell 3

In [8]:
DATA_DIR = "/content/drive/MyDrive/processed"

TRAIN_IMG_DIR = f"{DATA_DIR}/train/images"
TRAIN_MASK_DIR = f"{DATA_DIR}/train/masks"

VAL_IMG_DIR = f"{DATA_DIR}/val/images"
VAL_MASK_DIR = f"{DATA_DIR}/val/masks"

TEST_IMG_DIR = f"{DATA_DIR}/test/images"
TEST_MASK_DIR = f"{DATA_DIR}/test/masks"

IMAGE_SIZE = 512

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4

EPOCHS = 50
LR = 1e-4


PATIENCE = 8
MIN_DELTA = 1e-4

BASE_CHANNELS = 32

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAVE_DIR = Path("/content/drive/MyDrive/tomato_unetpp_checkpoints")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = SAVE_DIR / "best_unetpp_model.pth"
LAST_MODEL_PATH = SAVE_DIR / "last_unetpp_model.pth"
WEIGHTS_ONLY_PATH = SAVE_DIR / "unetpp_weights_only.pth"
HISTORY_PATH = SAVE_DIR / "unetpp_training_history.json"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Dataset:", DATA_DIR)
print("Device:", DEVICE)
print("Save dir:", SAVE_DIR)
print("Batch size:", BATCH_SIZE)
print("Grad accumulation:", GRAD_ACCUM_STEPS)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Base channels:", BASE_CHANNELS)

Dataset: /content/drive/MyDrive/processed
Device: cuda
Save dir: /content/drive/MyDrive/tomato_unetpp_checkpoints
Batch size: 2
Grad accumulation: 4
Effective batch size: 8
Base channels: 32


Cell 4

In [9]:
def pil_rgb_to_tensor(img):
    img = img.convert("RGB")
    w, h = img.size

    data = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
    tensor = data.view(h, w, 3).permute(2, 0, 1).float() / 255.0

    return tensor


def pil_mask_to_tensor(mask):
    mask = mask.convert("L")
    w, h = mask.size

    data = torch.ByteTensor(torch.ByteStorage.from_buffer(mask.tobytes()))
    tensor = data.view(h, w).float()

    tensor = (tensor > 127).float()
    tensor = tensor.unsqueeze(0)

    return tensor


class TomatoDataset(Dataset):
    def __init__(self, images_dir, masks_dir, image_size=512, train=True):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.image_size = image_size
        self.train = train

        exts = [".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG", ".BMP"]

        image_paths = [p for p in self.images_dir.glob("*") if p.suffix in exts]
        mask_paths = [p for p in self.masks_dir.glob("*") if p.suffix in exts]

        mask_map = {p.stem: p for p in mask_paths}

        self.pairs = []
        missing = 0

        for img_path in image_paths:
            if img_path.stem in mask_map:
                self.pairs.append((img_path, mask_map[img_path.stem]))
            else:
                missing += 1

        self.pairs = sorted(self.pairs, key=lambda x: x[0].name)

        print("Images found:", len(image_paths))
        print("Masks found :", len(mask_paths))
        print("Valid pairs :", len(self.pairs))
        print("Missing masks skipped:", missing)

        if len(self.pairs) == 0:
            raise RuntimeError("Không tìm thấy cặp image-mask hợp lệ.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = image.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        if self.train:
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
                mask = ImageOps.mirror(mask)

            if random.random() < 0.5:
                image = ImageOps.flip(image)
                mask = ImageOps.flip(mask)

            if random.random() < 0.5:
                k = random.choice([1, 2, 3])
                image = image.rotate(90 * k)
                mask = mask.rotate(90 * k)

        image = pil_rgb_to_tensor(image)
        mask = pil_mask_to_tensor(mask)

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

        image = (image - mean) / std

        return image, mask

Cell 5

In [10]:
train_dataset = TomatoDataset(
    TRAIN_IMG_DIR,
    TRAIN_MASK_DIR,
    image_size=IMAGE_SIZE,
    train=True
)

val_dataset = TomatoDataset(
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    image_size=IMAGE_SIZE,
    train=False
)

test_dataset = TomatoDataset(
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    image_size=IMAGE_SIZE,
    train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Train:", len(train_dataset), "samples")
print("Val  :", len(val_dataset), "samples")
print("Test :", len(test_dataset), "samples")

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

Images found: 5634
Masks found : 5618
Valid pairs : 5618
Missing masks skipped: 16
Images found: 1207
Masks found : 1219
Valid pairs : 1203
Missing masks skipped: 4
Images found: 1229
Masks found : 1210
Valid pairs : 1210
Missing masks skipped: 19
Train: 5618 samples
Val  : 1203 samples
Test : 1210 samples
Train batches: 2809
Val batches  : 602
Test batches : 605


Cell 6

In [11]:
images, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Masks :", masks.shape)
print("Image min/max:", images.min().item(), images.max().item())
print("Mask unique values:", torch.unique(masks))

/tmp/ipykernel_2511/3519658755.py:5: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  data = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))


Images: torch.Size([2, 3, 512, 512])
Masks : torch.Size([2, 1, 512, 512])
Image min/max: -2.0836544036865234 2.2216994762420654
Mask unique values: tensor([0., 1.])


Cell 7

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetPlusPlus(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32):
        super().__init__()

        nb_filter = [
            base_channels,
            base_channels * 2,
            base_channels * 4,
            base_channels * 8,
            base_channels * 16
        ]

        self.pool = nn.MaxPool2d(2, 2)

        self.conv0_0 = ConvBlock(in_channels, nb_filter[0])
        self.conv1_0 = ConvBlock(nb_filter[0], nb_filter[1])
        self.conv2_0 = ConvBlock(nb_filter[1], nb_filter[2])
        self.conv3_0 = ConvBlock(nb_filter[2], nb_filter[3])
        self.conv4_0 = ConvBlock(nb_filter[3], nb_filter[4])

        self.conv0_1 = ConvBlock(nb_filter[0] + nb_filter[1], nb_filter[0])
        self.conv1_1 = ConvBlock(nb_filter[1] + nb_filter[2], nb_filter[1])
        self.conv2_1 = ConvBlock(nb_filter[2] + nb_filter[3], nb_filter[2])
        self.conv3_1 = ConvBlock(nb_filter[3] + nb_filter[4], nb_filter[3])

        self.conv0_2 = ConvBlock(nb_filter[0] * 2 + nb_filter[1], nb_filter[0])
        self.conv1_2 = ConvBlock(nb_filter[1] * 2 + nb_filter[2], nb_filter[1])
        self.conv2_2 = ConvBlock(nb_filter[2] * 2 + nb_filter[3], nb_filter[2])

        self.conv0_3 = ConvBlock(nb_filter[0] * 3 + nb_filter[1], nb_filter[0])
        self.conv1_3 = ConvBlock(nb_filter[1] * 3 + nb_filter[2], nb_filter[1])

        self.conv0_4 = ConvBlock(nb_filter[0] * 4 + nb_filter[1], nb_filter[0])

        self.final = nn.Conv2d(nb_filter[0], out_channels, kernel_size=1)

    def upsample_to(self, x, target):
        return F.interpolate(
            x,
            size=target.shape[2:],
            mode="bilinear",
            align_corners=False
        )

    def forward(self, x):
        x0_0 = self.conv0_0(x)

        x1_0 = self.conv1_0(self.pool(x0_0))
        x0_1 = self.conv0_1(torch.cat([x0_0, self.upsample_to(x1_0, x0_0)], dim=1))

        x2_0 = self.conv2_0(self.pool(x1_0))
        x1_1 = self.conv1_1(torch.cat([x1_0, self.upsample_to(x2_0, x1_0)], dim=1))
        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.upsample_to(x1_1, x0_0)], dim=1))

        x3_0 = self.conv3_0(self.pool(x2_0))
        x2_1 = self.conv2_1(torch.cat([x2_0, self.upsample_to(x3_0, x2_0)], dim=1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.upsample_to(x2_1, x1_0)], dim=1))
        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.upsample_to(x1_2, x0_0)], dim=1))

        x4_0 = self.conv4_0(self.pool(x3_0))
        x3_1 = self.conv3_1(torch.cat([x3_0, self.upsample_to(x4_0, x3_0)], dim=1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.upsample_to(x3_1, x2_0)], dim=1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.upsample_to(x2_2, x1_0)], dim=1))
        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.upsample_to(x1_3, x0_0)], dim=1))

        output = self.final(x0_4)

        return output


model = UNetPlusPlus(
    in_channels=3,
    out_channels=1,
    base_channels=BASE_CHANNELS
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model: Custom U-Net++")
print("Total params:", total_params)
print("Trainable params:", trainable_params)
print("Device:", DEVICE)

Model: Custom U-Net++
Total params: 9159681
Trainable params: 9159681
Device: cuda


Cell 8

In [13]:
bce_loss = nn.BCEWithLogitsLoss()


def dice_loss_fn(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)

    intersection = torch.sum(probs * targets, dim=dims)
    cardinality = torch.sum(probs + targets, dim=dims)

    dice = (2.0 * intersection + eps) / (cardinality + eps)

    return 1.0 - dice.mean()


def loss_fn(logits, masks):
    return bce_loss(logits, masks) + dice_loss_fn(logits, masks)


def calculate_metrics(logits, masks, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    masks = masks.float()

    intersection = (preds * masks).sum(dim=(1, 2, 3))
    pred_sum = preds.sum(dim=(1, 2, 3))
    mask_sum = masks.sum(dim=(1, 2, 3))

    union = pred_sum + mask_sum - intersection

    iou = (intersection + eps) / (union + eps)
    dice = (2 * intersection + eps) / (pred_sum + mask_sum + eps)

    tp = intersection
    fp = pred_sum - intersection
    fn = mask_sum - intersection

    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)

    accuracy = (preds == masks).float().mean(dim=(1, 2, 3))

    return {
        "iou": iou.mean().item(),
        "dice": dice.mean().item(),
        "accuracy": accuracy.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item()
    }


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

print("Loss: BCEWithLogitsLoss + DiceLoss")
print("Optimizer: AdamW")
print("LR:", LR)

Loss: BCEWithLogitsLoss + DiceLoss
Optimizer: AdamW
LR: 0.0001


/tmp/ipykernel_2511/4191618991.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


In [14]:
class EarlyStopping:
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            self.counter = 0
            return True

        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            return True

        self.counter += 1

        if self.counter >= self.patience:
            self.should_stop = True

        return False


early_stopping = EarlyStopping(
    patience=PATIENCE,
    min_delta=MIN_DELTA
)

print("Early stopping patience:", PATIENCE)

Early stopping patience: 8


In [15]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    total_acc = 0.0
    total_precision = 0.0
    total_recall = 0.0

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(loader, desc="Training U-Net++", leave=False)

    for step, (images, masks) in enumerate(progress_bar):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits = model(images)
            loss = loss_fn(logits, masks)
            loss_backward = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_backward).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        metrics = calculate_metrics(logits.detach(), masks)

        total_loss += loss.item()
        total_iou += metrics["iou"]
        total_dice += metrics["dice"]
        total_acc += metrics["accuracy"]
        total_precision += metrics["precision"]
        total_recall += metrics["recall"]

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "iou": f"{metrics['iou']:.4f}",
            "dice": f"{metrics['dice']:.4f}"
        })

    n = len(loader)

    return {
        "loss": total_loss / n,
        "iou": total_iou / n,
        "dice": total_dice / n,
        "accuracy": total_acc / n,
        "precision": total_precision / n,
        "recall": total_recall / n
    }


@torch.no_grad()
def validate_one_epoch(model, loader):
    model.eval()

    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    total_acc = 0.0
    total_precision = 0.0
    total_recall = 0.0

    progress_bar = tqdm(loader, desc="Validation U-Net++", leave=False)

    for images, masks in progress_bar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits = model(images)
            loss = loss_fn(logits, masks)

        metrics = calculate_metrics(logits, masks)

        total_loss += loss.item()
        total_iou += metrics["iou"]
        total_dice += metrics["dice"]
        total_acc += metrics["accuracy"]
        total_precision += metrics["precision"]
        total_recall += metrics["recall"]

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "iou": f"{metrics['iou']:.4f}",
            "dice": f"{metrics['dice']:.4f}"
        })

    n = len(loader)

    return {
        "loss": total_loss / n,
        "iou": total_iou / n,
        "dice": total_dice / n,
        "accuracy": total_acc / n,
        "precision": total_precision / n,
        "recall": total_recall / n
    }

In [16]:
best_iou = 0.0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_iou": [],
    "val_iou": [],
    "train_dice": [],
    "val_dice": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "train_precision": [],
    "val_precision": [],
    "train_recall": [],
    "val_recall": [],
    "lr": []
}

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    print(f"\nU-Net++ Epoch [{epoch}/{EPOCHS}]")
    print("-" * 60)

    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = validate_one_epoch(model, val_loader)

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_metrics["iou"])

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])

    history["train_iou"].append(train_metrics["iou"])
    history["val_iou"].append(val_metrics["iou"])

    history["train_dice"].append(train_metrics["dice"])
    history["val_dice"].append(val_metrics["dice"])

    history["train_accuracy"].append(train_metrics["accuracy"])
    history["val_accuracy"].append(val_metrics["accuracy"])

    history["train_precision"].append(train_metrics["precision"])
    history["val_precision"].append(val_metrics["precision"])

    history["train_recall"].append(train_metrics["recall"])
    history["val_recall"].append(val_metrics["recall"])

    history["lr"].append(current_lr)

    elapsed = time.time() - start_time

    print(
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"IoU: {train_metrics['iou']:.4f} | "
        f"Dice: {train_metrics['dice']:.4f} | "
        f"Acc: {train_metrics['accuracy']:.4f}"
    )

    print(
        f"Val   Loss: {val_metrics['loss']:.4f} | "
        f"IoU: {val_metrics['iou']:.4f} | "
        f"Dice: {val_metrics['dice']:.4f} | "
        f"Acc: {val_metrics['accuracy']:.4f}"
    )

    print(
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f}"
    )

    print(f"LR: {current_lr:.8f}")
    print(f"Time: {elapsed:.2f}s")

    torch.save({
        "epoch": epoch,
        "model_name": "Custom U-Net++",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_iou": best_iou,
        "history": history,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "base_channels": BASE_CHANNELS,
        "lr": LR
    }, LAST_MODEL_PATH)

    improved = early_stopping.step(val_metrics["iou"])

    if improved:
        best_iou = val_metrics["iou"]

        torch.save({
            "epoch": epoch,
            "model_name": "Custom U-Net++",
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "base_channels": BASE_CHANNELS,
            "lr": LR
        }, BEST_MODEL_PATH)

        print(f"Best model saved: {BEST_MODEL_PATH}")
        print(f"Best Val IoU: {best_iou:.4f}")
    else:
        print(
            f"No improvement. Early stopping counter: "
            f"{early_stopping.counter}/{early_stopping.patience}"
        )

    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4)

    if early_stopping.should_stop:
        print("\nEarly stopping activated.")
        print(f"Stopped at epoch {epoch}.")
        print(f"Best Val IoU: {best_iou:.4f}")
        break

print("\nU-Net++ training finished.")
print("Best Val IoU:", best_iou)
print("Best model:", BEST_MODEL_PATH)


U-Net++ Epoch [1/50]
------------------------------------------------------------


Training U-Net++:   0%|          | 0/2809 [00:00<?, ?it/s]

/tmp/ipykernel_2511/1565592895.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Validation U-Net++:   0%|          | 0/602 [00:00<?, ?it/s]

/tmp/ipykernel_2511/1565592895.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


...

Validation U-Net++:   0%|          | 0/602 [00:00<?, ?it/s]

Train Loss: 0.6033 | IoU: 0.3960 | Dice: 0.4900 | Acc: 0.9934
Val   Loss: 0.6500 | IoU: 0.3866 | Dice: 0.4777 | Acc: 0.9914
Val Precision: 0.5880 | Val Recall: 0.6048
LR: 0.00002500
Time: 1015.95s
No improvement. Early stopping counter: 7/8

U-Net++ Epoch [22/50]
------------------------------------------------------------


Training U-Net++:   0%|          | 0/2809 [00:00<?, ?it/s]

Validation U-Net++:   0%|          | 0/602 [00:00<?, ?it/s]

Train Loss: 0.6025 | IoU: 0.4026 | Dice: 0.4966 | Acc: 0.9933
Val   Loss: 0.6546 | IoU: 0.3941 | Dice: 0.4841 | Acc: 0.9912
Val Precision: 0.6192 | Val Recall: 0.6033
LR: 0.00002500
Time: 1018.17s
No improvement. Early stopping counter: 8/8

Early stopping activated.
Stopped at epoch 22.
Best Val IoU: 0.3965

U-Net++ training finished.
Best Val IoU: 0.3964745350779108
Best model: /content/drive/MyDrive/tomato_unetpp_checkpoints/best_unetpp_model.pth


In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Loaded best model:", BEST_MODEL_PATH)
print("Best epoch:", checkpoint["epoch"])
print("Best Val IoU:", checkpoint["best_iou"])

test_metrics = validate_one_epoch(model, test_loader)

print("\nU-Net++ Test results")
print("-" * 40)
print("Loss     :", round(test_metrics["loss"], 4))
print("IoU      :", round(test_metrics["iou"], 4))
print("Dice     :", round(test_metrics["dice"], 4))
print("Accuracy :", round(test_metrics["accuracy"], 4))
print("Precision:", round(test_metrics["precision"], 4))
print("Recall   :", round(test_metrics["recall"], 4))

Loaded best model: /content/drive/MyDrive/tomato_unetpp_checkpoints/best_unetpp_model.pth
Best epoch: 14
Best Val IoU: 0.3964745350779108


Validation U-Net++:   0%|          | 0/605 [00:00<?, ?it/s]

/tmp/ipykernel_2511/1565592895.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):



U-Net++ Test results
----------------------------------------
Loss     : 0.6522
IoU      : 0.3894
Dice     : 0.4808
Accuracy : 0.9908
Precision: 0.6928
Recall   : 0.5276


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import torch

@torch.no_grad()
def show_predictions(model, loader, num_samples=4, threshold=0.5):
    # Lấy 1 batch dữ liệu
    images, masks = next(iter(loader))
    images = images.to(DEVICE)
    masks = masks.to(DEVICE)

    # Dự đoán
    logits = model(images)
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    # Đưa về CPU để vẽ
    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()

    # Thông số chuẩn hóa (dùng giống lúc bạn định nghĩa ở Dataset)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    n = min(num_samples, images.size(0))
    plt.figure(figsize=(12, 4 * n))

    for i in range(n):
        # Trả ảnh về hệ màu RGB gốc
        img = images[i].permute(1, 2, 0).numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)

        gt_mask = masks[i][0].numpy()
        pred_mask = preds[i][0].numpy()

        # Vẽ ảnh gốc
        plt.subplot(n, 3, i * 3 + 1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis("off")

        # Vẽ nhãn Ground Truth
        plt.subplot(n, 3, i * 3 + 2)
        plt.imshow(gt_mask, cmap="gray")
        plt.title("Ground Truth")
        plt.axis("off")

        # Vẽ nhãn Dự đoán của U-Net++
        plt.subplot(n, 3, i * 3 + 3)
        plt.imshow(pred_mask, cmap="gray")
        plt.title("U-Net++ Prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Gọi hàm để hiển thị ảnh từ tập validation (hoặc test_loader)
show_predictions(model, val_loader, num_samples=4, threshold=0.5)

ModuleNotFoundError: No module named 'matplotlib'